# Aula 5 — Regressão Linear com Múltiplos Regressores

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

O problema central do curso: quando um determinante de $Y$ correlacionado com $X$ fica
de fora, o estimador é **viesado**. Quatro resultados:

1. o **viés de variável omitida**, medido e conferido contra a fórmula;
2. ele **não desaparece** quando a amostra cresce — é erro de identificação, não ruído;
3. $R^2$ contra $\bar{R}^2$ diante de regressores inúteis;
4. o custo em precisão da **multicolinearidade imperfeita**.

A fórmula do viés:

$$\text{plim}\,\hat\beta_1^{\text{simples}} = \beta_1 + \beta_2\frac{\text{Cov}(X_1, X_2)}{\text{Var}(X_1)}$$

In [ ]:
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))
theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)

ep_robusto <- function(modelo, tipo = "HC1") {
  X <- model.matrix(modelo); u <- residuals(modelo)
  n <- nrow(X); k <- ncol(X)
  bread  <- solve(crossprod(X))
  ajuste <- if (tipo == "HC1") n / (n - k) else 1
  sqrt(diag(bread %*% (ajuste * crossprod(X * u)) %*% bread))
}

## 1. Os dados: um modelo em que conhecemos a verdade

$PctEL$ (proporção de alunos aprendendo inglês) é **correlacionada com $STR$** e é
**determinante da nota** — as duas condições do viés de variável omitida.

In [ ]:
set.seed(6205)
n_d <- 420
b0 <- 686.0; b1 <- -1.10; b2 <- -0.65   # parâmetros VERDADEIROS

STR   <- rnorm(n_d, mean = 19.64, sd = 1.89)
rho   <- 0.19
sd_el <- 18.3
PctEL <- pmax(0, 35.8 + rho * (sd_el / 1.89) * (STR - 19.64) +
                 rnorm(n_d, 0, sd_el * sqrt(1 - rho^2)))
u <- rnorm(n_d, 0, 14.4)
TestScore <- b0 + b1 * STR + b2 * PctEL + u

ca <- data.frame(TestScore, STR, PctEL)
round(cor(ca), 3)

## 2. O viés, medido

In [ ]:
m_simples <- lm(TestScore ~ STR, data = ca)           # OMITE PctEL
m_multi   <- lm(TestScore ~ STR + PctEL, data = ca)   # inclui

vies_previsto <- b2 * cov(ca$STR, ca$PctEL) / var(ca$STR)

round(c(beta1_verdadeiro  = b1,
        estimado_simples  = unname(coef(m_simples)[2]),
        estimado_multiplo = unname(coef(m_multi)[2]),
        vies_observado    = unname(coef(m_simples)[2]) - b1,
        vies_pela_formula = vies_previsto), 4)

A regressão simples atribui ao tamanho da turma parte do efeito que é, na verdade, da
composição linguística dos distritos.

Note que o viés observado **não bate exatamente** com o da fórmula. Não é erro: a
fórmula vale no limite (`plim`), e com $n = 420$ ainda há bastante ruído amostral. A
célula abaixo mostra os dois convergindo.

In [ ]:
convergencia <- t(sapply(c(420, 5000, 100000, 1000000), function(N) {
  set.seed(6205)
  x1 <- rnorm(N, 19.64, 1.89)
  x2 <- 35.8 + rho * (sd_el / 1.89) * (x1 - 19.64) +
        rnorm(N, 0, sd_el * sqrt(1 - rho^2))
  yy <- b0 + b1 * x1 + b2 * x2 + rnorm(N, 0, 14.4)
  c(n = N,
    vies_observado = unname(coef(lm(yy ~ x1))[2]) - b1,
    vies_formula   = b2 * cov(x1, x2) / var(x1))
}))
cbind(convergencia,
      diferenca = convergencia[, 2] - convergencia[, 3]) |> round(4)

## 3. O viés **não** desaparece com $n$

Esta é a diferença essencial entre **viés** e **ruído**. Aumentamos a amostra de 100
a 100.000 e observamos os dois estimadores.

In [ ]:
tamanhos <- c(100, 500, 2000, 10000, 50000, 100000)
set.seed(11)

resultado <- t(sapply(tamanhos, function(N) {
  x1 <- rnorm(N, 19.64, 1.89)
  x2 <- pmax(0, 35.8 + rho * (sd_el / 1.89) * (x1 - 19.64) +
                rnorm(N, 0, sd_el * sqrt(1 - rho^2)))
  yy <- b0 + b1 * x1 + b2 * x2 + rnorm(N, 0, 14.4)
  c(n = N,
    simples  = unname(coef(lm(yy ~ x1))[2]),
    multipla = unname(coef(lm(yy ~ x1 + x2))[2]))
}))
round(resultado, 4)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 3.8)
# empilhamos na mão, para não depender de tidyr (nem sempre presente no Colab)
res <- as.data.frame(resultado)
d <- rbind(
  data.frame(n = res$n, modelo = "simples",  estimativa = res$simples),
  data.frame(n = res$n, modelo = "multipla", estimativa = res$multipla)
)

ggplot(d, aes(n, estimativa, colour = modelo)) +
  geom_hline(yintercept = b1, linetype = 2, colour = vd, linewidth = 0.9) +
  geom_line(linewidth = 1) + geom_point(size = 2) +
  scale_x_log10() +
  scale_colour_manual(values = c(simples = vm, multipla = az)) +
  labs(x = "tamanho da amostra (escala log)", y = expression(hat(beta)[1]),
       colour = NULL,
       subtitle = "verde tracejado: valor verdadeiro") +
  theme(legend.position = "bottom")

A regressão múltipla **converge** para o valor verdadeiro; a simples converge para o
**valor errado**. Mais dados não corrigem um problema de identificação — apenas
estimam o número errado com mais precisão.

## 4. O sinal do viés é previsível

O viés é $\beta_2 \cdot \text{Cov}(X_1,X_2)/\text{Var}(X_1)$: o produto dos sinais
de $\beta_2$ e da correlação.

In [ ]:
set.seed(303)
cenarios <- expand.grid(sinal_b2 = c(-0.65, 0.65), sinal_rho = c(-0.5, 0.5))

out <- t(apply(cenarios, 1, function(p) {
  bb2 <- p[["sinal_b2"]]; rr <- p[["sinal_rho"]]
  x1 <- rnorm(20000, 19.64, 1.89)
  x2 <- 35.8 + rr * (sd_el / 1.89) * (x1 - 19.64) + rnorm(20000, 0, sd_el * sqrt(1 - rr^2))
  yy <- b0 + b1 * x1 + bb2 * x2 + rnorm(20000, 0, 14.4)
  c(beta2 = bb2, rho = rr,
    estimado = unname(coef(lm(yy ~ x1))[2]),
    vies = unname(coef(lm(yy ~ x1))[2]) - b1)
}))
round(out, 4)

Quando $\beta_2$ e $\rho$ têm o **mesmo** sinal, o viés é positivo; sinais opostos,
viés negativo. Conhecer os sinais já permite dizer a **direção** do erro, mesmo sem
os dados da variável omitida.

## 5. $R^2$ contra $\bar{R}^2$: o teste do ruído puro

Acrescentamos 30 regressores **inteiramente aleatórios**, sem relação alguma com $Y$.

In [ ]:
# Usamos uma amostra MENOR (n = 120) porque é aí que a correção por graus de
# liberdade morde: com n grande, 30 regressores inúteis quase não custam nada.
set.seed(6205)
n_p <- 120
x1p <- rnorm(n_p, 19.64, 1.89)
x2p <- rnorm(n_p, 35.8, 18.3)
cap <- data.frame(TestScore = b0 + b1 * x1p + b2 * x2p + rnorm(n_p, 0, 14.4),
                  STR = x1p, PctEL = x2p)

set.seed(808)
lixo <- as.data.frame(matrix(rnorm(n_p * 80), nrow = n_p))
names(lixo) <- paste0("ruido", 1:80)

comparacao <- t(sapply(c(0, 5, 15, 30, 60, 80), function(k) {
  dd  <- if (k == 0) cap else cbind(cap, lixo[, 1:k, drop = FALSE])
  fit <- lm(TestScore ~ ., data = dd)
  c(regressores_ruido = k,
    R2          = summary(fit)$r.squared,
    R2_ajustado = summary(fit)$adj.r.squared)
}))
round(comparacao, 4)

O contraste é nítido. O $R^2$ **sobe monotonicamente** — de 0,45 para 0,82 — enquanto
os regressores acrescentados são puro ruído, sem relação alguma com $Y$. É por isso
que ele **não serve** para escolher especificação: ele nunca cai.

O $\bar{R}^2$ **cai**, porque a correção $\frac{n-1}{n-k-1}$ penaliza cada grau de
liberdade gasto. Ele sinaliza corretamente que o modelo está piorando.

> Note que o efeito depende de $n$. Com os 420 distritos originais, 30 regressores
> inúteis quase não moveriam o $\bar{R}^2$ — a penalização só morde quando $k$ é
> grande **em relação a** $n$. Vale testar: troque `n_p <- 120` por `n_p <- 420`.

> Cuidado: $\bar{R}^2$ **melhor não significa modelo causalmente melhor**. O critério
> de inclusão é o do viés de variável omitida — conhecimento do problema —, não o
> ajuste. A Aula 6 desenvolve exatamente esse ponto.

## 6. O custo da multicolinearidade imperfeita

In [ ]:
set.seed(500)
correlacoes <- c(0, 0.5, 0.8, 0.9, 0.95, 0.99)

mc <- t(sapply(correlacoes, function(r) {
  x1 <- rnorm(500)
  x2 <- r * x1 + rnorm(500, 0, sqrt(1 - r^2))
  yy <- 2 + 1.5 * x1 + 1.0 * x2 + rnorm(500)
  fit <- lm(yy ~ x1 + x2)
  c(rho = r,
    ep_b1 = summary(fit)$coefficients[2, 2],
    fator_teorico = 1 / sqrt(1 - r^2))
}))
mc <- cbind(mc, inflacao_observada = mc[, "ep_b1"] / mc[1, "ep_b1"])
round(mc, 4)

Com $\rho = 0{,}95$ o erro-padrão é cerca de **três vezes** o do caso ortogonal, e a
inflação observada acompanha de perto o fator teórico $1/\sqrt{1-\rho^2}$.

> Multicolinearidade **não é viés**: os coeficientes continuam não viesados. O que se
> perde é **precisão**. É um problema de dados insuficientes, não de especificação
> errada.

## Para experimentar

1. Na seção 2, faça $\rho = 0$ (regressores não correlacionados) e verifique que o
   viés desaparece — mesmo com $\beta_2 \neq 0$.
2. Faça `b2 <- 0` mantendo $\rho \neq 0$: o viés também some. São **duas** condições, e
   basta uma falhar.
3. Na seção 6, aumente $n$ para 5000: a multicolinearidade dói menos. Por quê?

---

⬅️ [Aula 4](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/04-inferencia.ipynb) · [🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb) · ➡️ [**Aula 6 — Teste F**](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/06-teste-f.ipynb)